In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

len(documents)

72

In [2]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    query,
    num_results=5
)

search_results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [3]:
[doc["filename"] for doc in search_results]

['01-agentic-rag/lessons/14-agentic-loop.md',
 '01-agentic-rag/lessons/15-frameworks.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/11-agents-intro.md',
 '01-agentic-rag/lessons/16-other-frameworks.md']

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

openai_client = OpenAI()

INSTRUCTIONS = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the course lessons.
Use only the facts from the CONTEXT when answering the QUESTION.
If the answer is not found in the CONTEXT, say "I don't know."
""".strip()

PROMPT_TEMPLATE = """
QUESTION:
{question}

CONTEXT:
{context}
""".strip()


def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append("FILENAME: " + doc["filename"])
        lines.append(doc["content"])
        lines.append("")

    return "\n\n".join(lines).strip()


def build_prompt(question, search_results):
    context = build_context(search_results)

    return PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )


def rag(question, index, model="gpt-5.4-mini"):
    search_results = index.search(
        question,
        num_results=5
    )

    prompt = build_prompt(question, search_results)

    messages = [
        {"role": "developer", "content": INSTRUCTIONS},
        {"role": "user", "content": prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text, response.usage, search_results, prompt

In [5]:
question = "How does the agentic loop keep calling the model until it stops?"

answer_q3, usage_q3, results_q3, prompt_q3 = rag(
    question,
    index=index,
    model="gpt-5.4-mini"
)

print(answer_q3)
print("Input tokens:", usage_q3.input_tokens)

The loop keeps calling the model by checking whether the response has any function calls.

- It calls the model.
- If the model returns a `function_call`, the code runs the tool, appends the tool output to the message history, and sets `has_function_calls = True`.
- If there are no function calls in that turn, `has_function_calls` stays `False`.
- The `while True` loop breaks when `has_function_calls == False`.

So it keeps going until the model returns a final answer with no more tool calls.
Input tokens: 7132


In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

len(chunks)

295

In [7]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

chunk_search_results = chunk_index.search(
    question,
    num_results=5
)

[doc["filename"] for doc in chunk_search_results]

['01-agentic-rag/lessons/14-agentic-loop.md',
 '01-agentic-rag/lessons/14-agentic-loop.md',
 '01-agentic-rag/lessons/14-agentic-loop.md',
 '01-agentic-rag/lessons/15-frameworks.md',
 '01-agentic-rag/lessons/15-frameworks.md']

In [8]:
answer_q5, usage_q5, results_q5, prompt_q5 = rag(
    question,
    index=chunk_index,
    model="gpt-5.4-mini"
)

print(answer_q5)
print("Q3 input tokens:", usage_q3.input_tokens)
print("Q5 input tokens:", usage_q5.input_tokens)
print("Ratio:", usage_q3.input_tokens / usage_q5.input_tokens)

The loop keeps calling the model inside a `while True` loop. Each turn it:

1. Calls the model with the current `messages`
2. Checks `response.output` for any `function_call`
3. Runs those tool calls and appends the results back to `messages`
4. Sets `has_function_calls = True` if any tool was used
5. Stops only when `has_function_calls == False`

So the stopping condition is: if the model returns a turn with no function calls, the loop breaks and the final answer is returned.
Q3 input tokens: 7132
Q5 input tokens: 2316
Ratio: 3.079447322970639


In [9]:
import json

def search(query: str) -> list[dict]:
    """
    Search the course lesson chunks for relevant information.
    """
    return chunk_index.search(
        query,
        num_results=5
    )


search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the course lesson chunks for relevant information.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


agent_instructions = """
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
""".strip()


def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)
    else:
        result = {"error": f"Unknown function: {call.name}"}

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": json.dumps(result, indent=2)
    }


def agent_loop(question, model="gpt-5.4-mini", max_iterations=10):
    messages = [
        {"role": "developer", "content": agent_instructions},
        {"role": "user", "content": question}
    ]

    search_call_count = 0
    last_answer = None

    for it in range(1, max_iterations + 1):
        print(f"iteration #{it}")

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        has_function_calls = False

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)

                if item.name == "search":
                    search_call_count += 1

                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                last_answer = item.content[0].text
                print("ASSISTANT:")
                print(last_answer)

        if not has_function_calls:
            break

    return {
        "answer": last_answer,
        "search_call_count": search_call_count,
        "messages": messages
    }

In [10]:
agent_result = agent_loop(
    "How does the agentic loop work, and how is it different from plain RAG?",
    model="gpt-5.4-mini"
)

print("Search calls:", agent_result["search_call_count"])
print(agent_result["answer"])

iteration #1
function_call: search {"query":"agentic loop plain RAG difference course lesson"}
function_call: search {"query":"agentic loop iterative retrieve act reflect lesson"}
function_call: search {"query":"RAG vs agentic loop retrieval generation lesson"}
iteration #2
ASSISTANT:
The **agentic loop** is a 반복 cycle where the model can **decide what to do next** instead of answering in one shot.

### How the agentic loop works
From the lesson, the pattern is:

1. Send the user question to the LLM.
2. The LLM may return a **tool call** instead of a final answer.
3. Your code runs that tool, like `search`.
4. You send the tool result back to the LLM.
5. Repeat until the model produces a final answer with **no more tool calls**.

The lesson describes it as a `while` loop that:
- calls the LLM,
- executes any tool calls it returns,
- sends the results back,
- stops when the model is done.

It also keeps:
- **instructions** via a developer message,
- **tools** the agent can use,
- **memo

In [11]:
final_answers = {
    "Q1": "72",
    "Q2": "01-agentic-rag/lessons/14-agentic-loop.md",
    "Q3": "7000",
    "Q4": "295",
    "Q5": "3× fewer",
    "Q6": "4",
}

final_answers

{'Q1': '72',
 'Q2': '01-agentic-rag/lessons/14-agentic-loop.md',
 'Q3': '7000',
 'Q4': '295',
 'Q5': '3× fewer',
 'Q6': '4'}